In [25]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np 

In [2]:
loader = PyPDFLoader("data/10201979.pdf")
loader = PyPDFLoader("data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf")
pages = loader.load()

#### text Cleaning

In [3]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spacess
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    return cleaned_text

In [4]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

In [5]:
pages[: 10]

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience Academy;http://technoscienceacademy.com', 'keywords': 'IJSRCSEIT; http:/ijsrcseit.com', 'moddate': '2026-05-05T22:07:15+05:30', 'subject': 'Research Paper', 'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology', 'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Copyright © 202 6 The Author(s) : This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4.0/) International Journal of Scientific Research in Computer Science, Engineering and Information Technology Peer Reviewed and Refereed International Scientific Research Journal ISSN : 2456 3307 Available Online at : ijsrcseit.com do

#### Chunking
- splitting your text information into smaller sizes

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap= 100) # created an instance of the text splitter

In [7]:
chunks = splitter.split_documents(pages)

In [8]:
chunks[0:10]

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience Academy;http://technoscienceacademy.com', 'keywords': 'IJSRCSEIT; http:/ijsrcseit.com', 'moddate': '2026-05-05T22:07:15+05:30', 'subject': 'Research Paper', 'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology', 'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Copyright © 202 6 The Author(s) : This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4.0/) International Journal of Scientific Research in Computer Science,'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience

In [9]:
len(chunks)

405

### Create Embeddings:
- Create embeddings from our chunks
- perform a search:
    - get user input
    - calculate cosine similarity (check for similarity in text/ information)

In [10]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings

In [11]:
sample_text = "this is a blue car!"
print(createEmbedding(sample_text))

[[-0.010901414, -0.009878994, -0.10206994, 0.025146862, -0.029017122, -0.018902695, -0.026545098, -0.03951932, -0.021458616, 0.02708927, 0.021827163, -0.08983775, 0.037547894, 0.03862853, 0.02944821, -0.016738234, 0.022206597, -0.014421699, -0.013778293, 0.007764854, 0.065272786, -0.0096959155, 0.017373862, -0.06427167, 0.056987867, -0.0032896744, 0.031153461, 0.005081759, 0.042577192, -0.04700088, 0.015497484, -0.03952028, -0.057451714, 0.06289506, -0.07147578, 0.05663965, -0.021548716, 0.010057831, 0.004936667, 0.028589593, 0.006044981, 0.0025699749, 0.046583567, 0.058653835, 0.020789199, -0.01923741, 0.009776124, -0.011004501, -0.015905574, -0.015037399, 0.052893896, -0.078270756, 0.022161713, -0.08693023, 0.041561525, -0.017274125, 0.026195088, -0.043717206, 0.02374817, -0.037567746, 0.012711958, -0.00049154134, -0.000664973, -0.037378713, -0.04365364, -0.04755528, 0.03867042, -0.012658174, 0.029582467, 0.009293176, -0.039772976, 0.035374735, -0.024829699, 0.04648463, -0.018639453,

In [12]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [13]:
len(doc_embeddings)

405

- user asks a question
- embedd the question 
- check for similarity in the document  

In [14]:
user_query = input("ask me anything about this doc...")

In [15]:
print(user_query)

What is RAG ? 


In [16]:
user_query_embeddding = createEmbedding(user_query)

In [17]:
print(user_query_embeddding)

[[0.021800214, -0.06790617, -0.04095496, -0.051171307, 0.07879676, -0.05345957, -0.024368979, -0.0045479396, -0.062411156, 0.02734284, -0.01973811, -0.02740824, -0.009211044, -0.0015515457, -0.01644054, -0.03602845, 0.022694252, 0.031421952, 0.023124855, -0.033678405, -0.062806554, 0.03661201, -0.017592207, -0.030384379, 0.017066764, 0.019888481, -0.0669795, -0.015213726, 0.014561974, 0.047519367, -0.012768431, 0.014629245, -0.03601212, 0.028880259, -0.014809395, 0.0111329, -0.029225962, -0.0054492787, 0.0021810203, -0.017978055, 0.036704347, 0.044450536, 0.043368865, -0.0388506, -0.019332362, -0.06208902, -0.092337966, 0.006073782, 0.016862283, -0.018122248, 0.0012466994, -0.025531027, -0.016337082, -0.02591809, -0.059206236, -0.07514175, -0.029093046, -0.06984575, 0.0588018, -0.019687157, -0.031976547, -0.03513569, -0.04172092, 0.049233668, -0.037069783, 0.060355082, 0.052580245, 0.044329353, -0.021428708, 0.019973941, -0.059620544, -0.0038293756, -0.0036370172, 0.001993329, -0.00533

In [18]:
# - loop through the doc_embeedding 
# - calculate the cosine similarity score 
# - return the items with highest score 

above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embeddding)
    if score >= 0.6:
        result = (index, score)
        above_zero_point_six.append(result)
        # print(score, index)
    

In [19]:
above_zero_point_six

[(22, array([[0.61524955]])),
 (30, array([[0.61058265]])),
 (38, array([[0.62803633]])),
 (48, array([[0.62225883]])),
 (54, array([[0.68223846]])),
 (72, array([[0.61977187]])),
 (73, array([[0.60581567]])),
 (246, array([[0.68009279]])),
 (279, array([[0.60085262]]))]

In [20]:
largest =  max(above_zero_point_six, key=lambda x: x[1])
largest

(54, array([[0.68223846]]))

In [21]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

(RAG) RAG is a brand  new mixed system that claims to be able to fix the issues with pure generative models. There is a system that gets relevant papers or information from an outside source of


In [23]:
# (RAG) RAG is a brand  new mixed system that claims to be able to fix the issues with pure generative models. 
# There is a system that gets relevant papers or information from an outside source of

In [24]:
chunks[above_zero_point_six[2][0]].page_content

'and approaching human  level accuracy while remaining cost efficient. The results demonstrate that RAG provides essential support for applications that require a complete understanding of specialized'

#### intro to evaluations

In [31]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3])

In [32]:
# Dot Product Similarity
dot_sim = np.dot(a, b)

In [33]:
# Cosine Similarity
cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [34]:
# Euclidean Distance (Lower is better)
euclidean_dist = np.linalg.norm(a - b)   

In [35]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

 
 DOT PRODUCT SIM_SCORE: 14  
 
 COSINE SIM_SCORE: 1.0  
 
 EUCLIDEAN DISTANCE: 0.0  
 



In [53]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [37]:
text_one  = "I am driving a blue car"
text_two = "A blue car is being driven by me"

In [39]:
text_one_embeding = createEmbedding(text=text_one)
text_two_embeding = createEmbedding(text=text_two)

In [62]:
np.array(text_one_embeding[0][0:10])

array([ 0.01455792, -0.0139812 , -0.08144961, -0.00092482, -0.02538994,
       -0.03063966, -0.02463491, -0.03938487, -0.00139509,  0.0101542 ])

In [41]:
print(text_one_embeding)
print(text_two_embeding)

[[0.014557918, -0.013981196, -0.081449606, -0.0009248153, -0.025389938, -0.030639656, -0.024634914, -0.039384868, -0.0013950891, 0.010154196, 0.025915336, -0.081142195, -0.0059265173, 0.035093665, 0.031422313, 0.030813755, 0.01823106, 0.03014354, 0.012382759, -0.018471112, 0.025488988, 0.038547926, 0.00057397183, -0.028115116, 0.07139039, -0.011789485, 0.032748297, 0.015907325, 0.03632926, -0.040933654, 0.04211532, -0.014560291, -0.072398104, 0.07264557, -0.0075238612, 0.045316815, -0.03327973, -0.055124898, 0.009934046, 0.014962451, 0.0011419852, 0.025773346, 0.059193287, 0.00080225564, 0.032153726, -0.016031051, 0.01877644, 0.029738357, -0.0020366868, -0.03113033, 0.018945295, -0.08350322, -0.0011875832, -0.039620806, 0.03302715, -0.010333018, 0.005548194, -0.06573134, 0.061073706, -0.028520063, 0.003950013, 0.025559636, -0.01016838, -0.029128686, -0.044582937, -0.048252117, -0.0024905931, -0.009117223, -0.034856677, 0.03853163, -0.0033841939, -0.001177449, -0.04428975, 0.02568356, -

In [54]:
similarityChecks(text_one_embeding= text_one_embeding[0], text_two_embeding=text_two_embeding[0])

  DOT PRODUCT SIM_SCORE: 0.8640598935855981  
  COSINE SIM_SCORE: 0.8640602802857031  
  EUCLIDEAN DISTANCE: 0.5214204807564335  
 


In [ ]:
user_query = input("ask me anything...")

In [ ]:
    
    """
    - Ask the user:
        - would you like ot add a document? 
        - enter your query based on the document
        - close program
    - Add document then append it to my embeddings list 
    - Take users query -> embeddings -> perform a search in the embedding list 
    - return a score or top k possible responses

    """